# Week 7: Cohort Economics — The Truth Behind Your Growth Story
## Retention Curves, NRR, and Cohort Triangles

**Masterclass:** [Week 7 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week7_cohort_economics.html)

**Dataset:** UCI Online Retail II (download from UCI) or CDNOW (bundled)

**What You'll Build:**
1. Cohort assignment from raw transaction data
2. Retention triangle (heatmap)
3. Revenue cohort analysis with NRR (Net Revenue Retention)
4. Cohort-over-cohort comparison: are newer cohorts healthier?

In [ ]:
!pip install lifetimes pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifetimes.datasets import load_transaction_data
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Build Cohort Triangle from Transactions

In [ ]:
txns = load_transaction_data()
txns['date'] = pd.to_datetime(txns['date'])
print(f"Transactions: {len(txns):,} | Customers: {txns['id'].nunique():,}")

# Assign cohort = month of first purchase
first_purchase = txns.groupby('id')['date'].min().dt.to_period('M').rename('cohort')
txns = txns.merge(first_purchase, on='id')
txns['period'] = txns['date'].dt.to_period('M')

# Cohort index: months since first purchase
txns['cohort_index'] = (txns['period'] - txns['cohort']).apply(lambda x: x.n)

# Active customers per cohort per period
cohort_data = txns.groupby(['cohort', 'cohort_index'])['id'].nunique().reset_index()
cohort_data.columns = ['cohort', 'cohort_index', 'customers']

# Pivot to triangle
cohort_counts = cohort_data.pivot(index='cohort', columns='cohort_index', values='customers')
# Get cohort sizes (month 0)
cohort_sizes = cohort_counts[0]

# Retention rates
retention = cohort_counts.divide(cohort_sizes, axis=0)
print(f"Cohorts: {len(retention)}")
print(f"Max periods tracked: {retention.columns.max()}")
retention.head()

In [ ]:
# Retention heatmap
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(retention.iloc[:12, :12], annot=True, fmt='.0%', cmap='YlOrRd_r',
            ax=ax, vmin=0, vmax=0.5, linewidths=0.5)
ax.set_title('Customer Retention by Cohort (% still active)', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (Month of First Purchase)')
plt.tight_layout()
plt.show()

---
## Part 2: Revenue Cohort Analysis

In [ ]:
# Revenue per cohort per period
rev_data = txns.groupby(['cohort', 'cohort_index'])['spent'].sum().reset_index()
rev_pivot = rev_data.pivot(index='cohort', columns='cohort_index', values='spent')

# NRR: revenue in period t / revenue in period 0
nrr = rev_pivot.divide(rev_pivot[0], axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(nrr.iloc[:12, :12], annot=True, fmt='.0f', cmap='RdYlGn',
            ax=ax, center=100, vmin=0, vmax=200, linewidths=0.5)
ax.set_title('Net Revenue Retention by Cohort (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort')
plt.tight_layout()
plt.show()

In [ ]:
# Average retention curve across all cohorts
avg_retention = retention.mean()
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(avg_retention.index, avg_retention.values, 'o-', color='#264653', linewidth=2, markersize=6)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.1, color='#264653')
ax.set_title('Average Retention Curve (All Cohorts)', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Retention Rate')
ax.set_ylim(0, 1)
ax.axhline(y=avg_retention.iloc[1:].mean(), color='#c45d3e', linestyle='--',
           label=f'Avg post-M0: {avg_retention.iloc[1:].mean():.1%}')
ax.legend()
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Cohort Quality Trend
Compare the 3-month retention rate of early cohorts vs. late cohorts. Are newer cohorts healthier or weaker?

### Exercise 2: Revenue Per Customer
Build a cohort triangle of revenue-per-active-customer (not total revenue). This removes the size effect and shows true monetization trends.

### Exercise 3: Predict LTV from Early Retention
Using month-1 and month-2 retention rates as features, can you predict 6-month cumulative revenue per cohort? Fit a simple regression.

---
## Key Takeaways
1. **Cohort triangles** reveal what aggregate metrics hide
2. **NRR > 100%** means existing customers are spending more over time (expansion revenue)
3. **Average retention curves** can mask improving or deteriorating cohort quality
4. **Early retention** (month 1-2) is the strongest predictor of long-term cohort health